In [ ]:
import glob, os, shutil, subprocess, tempfile
from pathlib import Path
OCTAVE = shutil.which("octave-cli") or shutil.which("octave")
if OCTAVE is None and os.name == "nt":
    candidates = glob.glob(r"C:\Program Files\GNU Octave\Octave-*\mingw64\bin\octave-cli.exe")
    if candidates:
        OCTAVE = sorted(candidates)[-1]
if OCTAVE is None:
    raise RuntimeError("Octave CLI not found. Add octave-cli to PATH or install GNU Octave.")
WORK = Path(tempfile.mkdtemp(prefix="me256-coursework-"))
print(f"Octave: {OCTAVE}")
print(f"Temporary source directory: {WORK}")

for relative in ['cylindrical-mirror', 'plane-mirror', 'plane-refraction', 'reflection-geometry', 'spherical-mirror']:
    (WORK / relative).mkdir(parents=True, exist_ok=True)


# ME256 Homework 2: reflection and refraction

**Official statement:** [ME256 Homework 2 (2011)](https://mecheng.iisc.ac.in/suresh/me256/hw2011/homework2.html). The outline below paraphrases the tasks for study; use the linked page for the complete statement and exact requirements.

The assignment asks you to formulate and solve four geometric-optics models, then use sufficient-condition analysis to determine whether each computed stationary point is a local minimum, local maximum, or saddle point:

1. A ray reflecting from a straight line between two points in a plane.
2. Reflection from a concave spherical mirror between two points in space.
3. Reflection from a right circular cylinder between two points in space.
4. Refraction between two media, with the endpoints in a plane and on opposite sides of the interface.

The archived MATLAB programs mainly explore path-length/travel-time objectives, numerical candidate points, and plots. They do **not** by themselves provide all the stationary-point classification and sufficient-condition reasoning requested by the assignment; treat those as study questions to complete alongside the code.

## Study prompts

- Which coordinates parameterize the point where a ray meets each surface, and what feasible set do those coordinates have?
- What objective represents optical path length or travel time in each model?
- How do you use first- and second-order conditions to classify a stationary point, including possible competing extrema?

## Reproducibility

This notebook uses a Python 3 Jupyter kernel to materialize the embedded MATLAB source into a temporary directory, then invokes Octave CLI. The `.m` sources are embedded below; they are not stored separately in this repository. Install Jupyter/IPython and an Octave installation with the `optim` package. MATLAB with Optimization Toolbox remains the closest match to the original runtime.

### Review note

The maximum-path helper `cylindrical-mirror/reflection_cylinder1.m` contains a suspicious endpoint-coordinate term. It is intentionally preserved and flagged in the assignment README; compare it with the original code PDF before drawing conclusions. A corrected experiment, if desired, should be a separate variant.

## Embedded source code

Each following code cell contains one MATLAB source file and writes it into a temporary working directory. Run the notebook from top to bottom; no reconstructed `.m` files are required in the repository.

### `reflection-geometry/geometry1.m`

In [ ]:
%%writefile {WORK / 'reflection-geometry' / 'geometry1.m'}
% Reconstructed from the assignment2 source-code PDF.
% Problem-solving approach is retained; see the assignment README.
% Sweep points on a circular mirror and compare path length and ray angles.
clc;clear all;close all;
theta=[0:pi/100:49*pi/100 51*pi/100:pi/100:149*pi/100 151*pi/100:pi/100:2*pi];
x=[.3;-.5];                             %Ray emerges from this point
y=[.5;.5];                              %Ray reflects to this point
a=1;
p=[a*cos(theta);a*sin(theta)];          %Point on periphery
d=sqrt(x(1)^2+x(2)^2+a^2-2*a*(x(1)*cos(theta)+x(2)*sin(theta)))+sqrt(y(1)^2+y(2) ^2+a^2-2*a*(y(1)*cos(theta)+y(2)*sin(theta)));
%Distance traversed by ray
plot(theta*180/pi,d*50)                 %Law of Reflection holds when d is stationary w.r.to theta
%d upscaled to prominence in graph
hold on
m1=(x(2)-a*sin(theta))./(x(1)-a*cos(theta));
m2=(y(2)-a*sin(theta))./(y(1)-a*cos(theta));
m=tan(theta);
i=(atan((m1-m)./(1+m1.*m)));            %Angle of Incidence
r=(atan((m2-m)./(1+m2.*m)));            %Angle of Reflection
plot(theta*180/pi,i*180/pi,'r')
hold on
plot(theta*180/pi,r*180/pi,'g')
hold on
plot(theta*180/pi,(i+r)*180/pi,'c')     %Law of Reflection holdss when (i+r)=0
grid on
xlabel('theta');
legend('distance(x50)','Angle of Incidence','Angle of Reflection','i+r');


### `plane-mirror/Reflection_PlaneMirror_2D.m`

In [ ]:
%%writefile {WORK / 'plane-mirror' / 'Reflection_PlaneMirror_2D.m'}
% Reconstructed from the assignment2 source-code PDF.
% Problem-solving approach is retained; see the assignment README.
% Find a reflection point on a straight mirror by minimizing total path length.
clc;clear all;close all;
global x y
m=2;
c=3;
M=@(x,y) y-m*x-c;                        %Equation of Mirror
x=[1;2];                                 %Ray emerges from this point
y=[3;7];                                 %Ray reflected to this point
z0=[0;c];
if(x(2)-m*x(1)-c)*(y(2)-m*y(1)-c)<0
    display('Points are opposite sides of mirror; Reflection is not possible.')
else
    [z,fval,exitflag]=fmincon(@reflection,z0,[],[],[-m 1],c,[],[],[],optimset('Algorithm','active-set'))
    xmin=min(x(1),y(1))-2;
    xmax=max(x(1),y(1))+2;
    ymin=min(x(2),y(2))-2;
    ymax=max(x(2),y(2))+2;
    ezplot(M,[xmin,xmax,ymin,ymax])
    hold on
    line([x(1),z(1),y(1)],[x(2),z(2),y(2)],'Color','r')
    plot(x(1),x(2),'Marker','o')
    text(x(1)+.15,x(2),'x');
    plot(y(1),y(2),'Marker','o')
    text(y(1)+.15,y(2),'y');
    plot(z(1),z(2),'Marker','o')
    text(z(1)+.15,z(2),'z');
    legend('Mirror','Reflection Path x->z->y')
    title('Reflection on Plane Mirror')
end
incidence_angle=(atan(-1/m)-atan((x(2)-z(2))/(x(1)-z(1))))*180/pi;
reflection_angle=(atan((y(2)-z(2))/(y(1)-z(1)))-atan(-1/m))*180/pi;
if incidence_angle<0
    incidence_angle=incidence_angle+180;
end
if reflection_angle<0
    reflection_angle=reflection_angle+180;
end
incidence_angle
reflection_angle


### `plane-mirror/reflection.m`

In [ ]:
%%writefile {WORK / 'plane-mirror' / 'reflection.m'}
% Reconstructed from the assignment2 source-code PDF.
% Problem-solving approach is retained; see the assignment README.
% Objective: sum the distances from the two ray endpoints to the mirror point.
function d=reflection(z)
    global x y
    d=norm(x-z,2)+norm(y-z,2);


### `spherical-mirror/Reflection_SphericalMirror.m`

In [ ]:
%%writefile {WORK / 'spherical-mirror' / 'Reflection_SphericalMirror.m'}
% Reconstructed from the assignment2 source-code PDF.
% Problem-solving approach is retained; see the assignment README.
% Compare minimum- and maximum-path reflection points on a circular mirror.
clc;clear all;close all;
global x y a
a=1;
M=@(x,y) x^2+y^2-a^2;                %Equation of Spherical Mirror
%Origin set at the centre
x=[-.5;.7];                          %Ray emerges from this point
y=[.5;.5];                           %Ray reflected to this point
theta0=1;
if x(1)^2+x(2)^2>a^2
    display('Point x is outside of the mirror')
else
    if y(1)^2+y(2)^2>a^2
        display('Point y is outside of the mirror')
    else
        [theta_dmin,dmin,exitflag]=fmincon(@reflection_spherical,theta0,[],[],[], [],-pi,pi,[],optimset('Algorithm','active-set'))
        %Reflection corresponding to min{d}
        [theta_dmax,dmax_neg,exitflag]=fmincon(@reflection_spherical1,theta0,[], [],[],[],-pi,pi,[],optimset('Algorithm','active-set'))
        %Reflection corresponding to max{d}
        z_dmin=[a*cos(theta_dmin);a*sin(theta_dmin)];
        z_dmax=[a*cos(theta_dmax);a*sin(theta_dmax)];
        ezplot(M)
        hold on
        line([x(1),z_dmin(1),y(1)],[x(2),z_dmin(2),y(2)],'Color','r')
        hold on
        line([x(1),z_dmax(1),y(1)],[x(2),z_dmax(2),y(2)],'Color','c')
        plot(x(1),x(2),'Marker','o')
        text(x(1)+a/20,x(2),'x');
        plot(y(1),y(2),'Marker','o')
        text(y(1)+a/20,y(2),'y');
        plot(z_dmin(1),z_dmin(2),'Marker','o')
        text(z_dmin(1)+a/20,z_dmin(2),'z_dmin');
        plot(z_dmax(1),z_dmax(2),'Marker','o')
        text(z_dmax(1)+a/20,z_dmax(2),'z_dmax');
        axis square
        axis tight
        xlabel('x')
        ylabel('y')
        legend('Mirror','Reflection x>dmin>y','Reflection x>dmax>y')
        title('Reflection on Concave Spherical Mirror')
        incindent_angle_dmin=angle((x(1)-z_dmin(1)+(x(2)-z_dmin(2))*i)/(-z_dmin(1) -z_dmin(2)*i))*180/pi
        reflection_angle_dmin=angle((-z_dmin(1)-z_dmin(2)*i)/(y(1)-z_dmin(1)+(y(2) -z_dmin(2))*i))*180/pi
        incindent_angle_dmax=angle((x(1)-z_dmax(1)+(x(2)-z_dmax(2))*i)/(-z_dmax(1) -z_dmax(2)*i))*180/pi
        reflection_angle_dmax=angle((-z_dmax(1)-z_dmax(2)*i)/(y(1)-z_dmax(1)+(y(2) -z_dmax(2))*i))*180/pi
    end
end


### `spherical-mirror/reflection_spherical.m`

In [ ]:
%%writefile {WORK / 'spherical-mirror' / 'reflection_spherical.m'}
% Reconstructed from the assignment2 source-code PDF.
% Problem-solving approach is retained; see the assignment README.
% Total path length for a reflection point parameterized by its angle.
function f=reflection_spherical(theta)
    global x y a
    f=sqrt(x(1)^2+x(2)^2+a^2-2*a*(x(1)*cos(theta)+x(2)*sin(theta)))+sqrt(y(1)^2+y(2) ^2+a^2-2*a*(y(1)*cos(theta)+y(2)*sin(theta)));


### `spherical-mirror/reflection_spherical1.m`

In [ ]:
%%writefile {WORK / 'spherical-mirror' / 'reflection_spherical1.m'}
% Reconstructed from the assignment2 source-code PDF.
% Problem-solving approach is retained; see the assignment README.
% Negated path length lets fmincon seek a maximum path length.
function f=reflection_spherical1(theta)
    global x y a
    f=-sqrt(x(1)^2+x(2)^2+a^2-2*a*(x(1)*cos(theta)+x(2)*sin(theta)))-sqrt(y(1)^2+y(2) ^2+a^2-2*a*(y(1)*cos(theta)+y(2)*sin(theta)));


### `cylindrical-mirror/Reflection_Cylindrical.m`

In [ ]:
%%writefile {WORK / 'cylindrical-mirror' / 'Reflection_Cylindrical.m'}
% Reconstructed from the assignment2 source-code PDF.
% Problem-solving approach is retained; see the assignment README.
% Optimize reflection points on a finite cylindrical mirror.
clc;clear all;close all;
global x y a
a=1;
x=[-.5;.1;.1];                          %Ray emerges from this point
y=[.5;-.5;.3];                           %Ray reflected to this point
z0=[1;.2];
if x(1)^2+x(2)^2>a^2
    display('Point x is outside of the mirror')
else
    if y(1)^2+y(2)^2>a^2
        display('Point y is outside of the mirror')
    else
        [z_dmin,dmin,exitflag]=fmincon(@reflection_cylinder,z0,[],[],[],[],[0;min(x(3),y(3))],[2*pi;max(x(3),y(3))],[],optimset('Algorithm','active-set'))
        %Reflection corresponding to min{d}
        [z_dmax,dmax_neg,exitflag]=fmincon(@reflection_cylinder1,z0,[],[],[],[],[- pi;min(x(3),y(3))],[pi;max(x(3),y(3))],[],optimset('Algorithm','active-set'))
        %Reflection corresponding to max{d}
        [X,Y,Z] = cylinder(a,200);
        Z(1,:)=min(x(3),y(3));
        Z(2,:)=max(x(3),y(3));
        surf(X,Y,Z)
        colormap winter
        shading interp
        if ~exist('OCTAVE_VERSION', 'builtin') || exist('alpha', 'file')
            alpha(.3)
        end
        hold on
        line([x(1),a*cos(z_dmin(1)),y(1)],[x(2),a*sin(z_dmin(1)),y(2)],[x(3), z_dmin(2),y(3)],'Color','r')
        hold on
        line([x(1),a*cos(z_dmax(1)),y(1)],[x(2),a*sin(z_dmax(1)),y(2)],[x(3), z_dmax(2),y(3)],'Color','y')
        plot3(x(1),x(2),x(3),'Marker','o')
        text(x(1)+a/20,x(2),x(3),'x');
        plot3(y(1),y(2),y(3),'Marker','o')
        text(y(1)+a/20,y(2),y(3),'y');
        plot3(a*cos(z_dmin(1)),a*sin(z_dmin(1)),z_dmin(2),'Marker','o')
        text(a*cos(z_dmin(1))+a/20,a*sin(z_dmin(1)),z_dmin(2),'z_dmin');
        plot3(a*cos(z_dmax(1)),a*sin(z_dmax(1)),z_dmax(2),'Marker','o')
        text(a*cos(z_dmax(1))+a/20,a*sin(z_dmax(1)),z_dmax(2),'z_dmax');
        axis square
        axis tight
        xlabel('x')
        ylabel('y')
        zlabel('z')
        legend('Mirror','Reflection x>z_dmin>y','Reflection x>z_dmax>y')
        title('Reflection on Cylindrical Mirror')
        incidence_angle_dmin=subspace([(x(1)-a*cos(z_dmin(1)));(x(2)-a*sin(z_dmin(1)));(x(3)-z_dmin(2))],[a*cos(z_dmin(1));a*sin(z_dmin(1));0])*180/pi
        reflection_angle_dmin=subspace([a*cos(z_dmin(1));a*sin(z_dmin(1));0],[(y(1)-a*cos(z_dmin(1)));(y(2)-a*sin(z_dmin(1)));(y(3)-z_dmin(2))])*180/pi
        incidence_angle_dmax=subspace([(x(1)-a*cos(z_dmax(1)));(x(2)-a*sin(z_dmax(1)));(x(3)-z_dmax(2))],[a*cos(z_dmax(1));a*sin(z_dmax(1));0])*180/pi
        reflection_angle_dmax=subspace([a*cos(z_dmax(1));a*sin(z_dmax(1));0],[(y(1)-a*cos(z_dmax(1)));(y(2)-a*sin(z_dmax(1)));(y(3)-z_dmax(2))])*180/pi
    end
end


### `cylindrical-mirror/Reflection_Cylindrical_Extern.m`

In [ ]:
%%writefile {WORK / 'cylindrical-mirror' / 'Reflection_Cylindrical_Extern.m'}
% Reconstructed from the assignment2 source-code PDF.
% Problem-solving approach is retained; see the assignment README.
% Repeat the cylindrical-mirror search with endpoints outside the cylinder.
clc;clear all;close all;
global x y a
a=1;
x=[5;7;.1];                          %Ray emerges from this point
y=[0;-5;.3];                           %Ray reflected to this point
z0=[1;.2];
if x(1)^2+x(2)^2<a^2
    display('Point x is inside of the mirror')
else
    if y(1)^2+y(2)^2<a^2
        display('Point y is inside of the mirror')
    else
        [z_dmin,dmin,exitflag]=fmincon(@reflection_cylinder,z0,[],[],[],[],[-pi; min(x(3),y(3))],[pi;max(x(3),y(3))],[],optimset('Algorithm','active-set'))
        %Reflection corresponding to min{d}
        [z_dmax,dmax_neg,exitflag]=fmincon(@reflection_cylinder1,z0,[],[],[],[],[- pi;min(x(3),y(3))],[pi;max(x(3),y(3))],[],optimset('Algorithm','active-set'))
        %Reflection corresponding to max{d}
        [X,Y,Z] = cylinder(a,200);
        Z(1,:)=min(x(3),y(3));
        Z(2,:)=max(x(3),y(3));
        surf(X,Y,Z)
        colormap copper
        shading interp
        if ~exist('OCTAVE_VERSION', 'builtin') || exist('alpha', 'file')
            alpha(.5)
        end
        hold on
        line([x(1),a*cos(z_dmin(1)),y(1)],[x(2),a*sin(z_dmin(1)),y(2)],[x(3), z_dmin(2),y(3)],'Color','r')
        hold on
        line([x(1),a*cos(z_dmax(1)),y(1)],[x(2),a*sin(z_dmax(1)),y(2)],[x(3), z_dmax(2),y(3)],'Color','g')
        plot3(x(1),x(2),x(3),'Marker','o')
        text(x(1)+a/20,x(2),x(3),'x');
        plot3(y(1),y(2),y(3),'Marker','o')
        text(y(1)+a/20,y(2),y(3),'y');
        plot3(a*cos(z_dmin(1)),a*sin(z_dmin(1)),z_dmin(2),'Marker','o')
        text(a*cos(z_dmin(1))+a/20,a*sin(z_dmin(1)),z_dmin(2),'z_dmin');
        plot3(a*cos(z_dmax(1)),a*sin(z_dmax(1)),z_dmax(2),'Marker','o')
        text(a*cos(z_dmax(1))+a/20,a*sin(z_dmax(1)),z_dmax(2),'z_dmax');
        set(gca,'DataAspectRatio',[1 1 .02])
        xlabel('x')
        ylabel('y')
        zlabel('z')
        legend('Mirror','Reflection x>z_dmin>y','Reflection x>z_dmax>y')
        title('Reflection on Cylindrical Mirror')
        incident_angle_dmin=subspace([(x(1)-a*cos(z_dmin(1)));(x(2)-a*sin(z_dmin(1)));(x(3)-z_dmin(2))],[a*cos(z_dmin(1));a*sin(z_dmin(1));0])*180/pi
        reflection_angle_dmin=subspace([a*cos(z_dmin(1));a*sin(z_dmin(1));0],[(y(1)-a*cos(z_dmin(1)));(y(2)-a*sin(z_dmin(1)));(y(3)-z_dmin(2))])*180/pi
        incident_angle_dmax=subspace([(x(1)-a*cos(z_dmax(1)));(x(2)-a*sin(z_dmax(1)));(x(3)-z_dmax(2))],[a*cos(z_dmax(1));a*sin(z_dmax(1));0])*180/pi
        reflection_angle_dmax=subspace([a*cos(z_dmax(1));a*sin(z_dmax(1));0],[(y(1)-a*cos(z_dmax(1)));(y(2)-a*sin(z_dmax(1)));(y(3)-z_dmax(2))])*180/pi
    end
end


### `cylindrical-mirror/reflection_cylinder.m`

In [ ]:
%%writefile {WORK / 'cylindrical-mirror' / 'reflection_cylinder.m'}
% Reconstructed from the assignment2 source-code PDF.
% Problem-solving approach is retained; see the assignment README.
% Sum the two endpoint-to-mirror distances for a cylindrical reflection.
function f=reflection_cylinder(z)
    global x y a
    f=sqrt(x(1)^2+x(2)^2+a^2-2*a*(x(1)*cos(z(1))+x(2)*sin(z(1)))+(x(3)-z(2))^2)+sqrt(y(1)^2+y(2)^2+a^2-2*a*(y(1)*cos(z(1))+y(2)*sin(z(1)))+(y(3)-z(2))^2);


### `cylindrical-mirror/reflection_cylinder1.m`

In [ ]:
%%writefile {WORK / 'cylindrical-mirror' / 'reflection_cylinder1.m'}
% Reconstructed from the assignment2 source-code PDF.
% Problem-solving approach is retained; see the assignment README.
% Reciprocal-distance objective used for the maximum-path search.
function f=reflection_cylinder1(z)
    global x y a
    g=sqrt(x(1)^2+x(2)^2+a^2-2*a*(x(1)*cos(z(1))+x(2)*sin(z(1)))+(x(3)-z(2))^2)+sqrt(y(1)^2+y(2)^2+a^2-2*a*(y(1)*cos(z(1))+y(2)*sin(z(1)))+(x(3)-z(2))^2);
    f=1/g;


### `plane-refraction/refraction_plane.m`

In [ ]:
%%writefile {WORK / 'plane-refraction' / 'refraction_plane.m'}
% Reconstructed from the assignment2 source-code PDF.
% Problem-solving approach is retained; see the assignment README.
% Minimize travel time across a planar interface between two media.
clc;clear all;close all;
global x y cx cy
cpos=1;                                  %Speed of Light at +ve side of plane of refraction
cneg=2;                                 %Speed of Light at -ve side of plane of refraction
m=0;
c=0;
M=@(x,y) y-m*x-c;                        %Equation of Plane of Refraction
x=[6;6];                                 %Ray emerges from this point
y=[-5;-5];                                %Ray refracted to this point
m1=(x(2)-y(2))/(x(1)-y(1));
c1=(x(1)*y(2)-x(2)*y(1))/(x(1)-y(1));
z0=[-m,1;-m1,1]\[c;c1];
sgnx=x(2)-m*x(1)-c;
sgny=y(2)-m*y(1)-c;
if sgnx*sgny>0
    display('Points are same sides of mirror; Refraction is not possible.')
else
    if sgnx>0
        cx=cpos;cy=cneg;
    else
        cx=cneg;cy=cpos;
    end
    [z,fval,exitflag]=fmincon(@refraction,z0,[],[],[-m 1],c,[],[],[],optimset('Algorithm','active-set'))
    xmin=min(x(1),y(1))-2;
    xmax=max(x(1),y(1))+2;
    ymin=min(x(2),y(2))-2;
    ymax=max(x(2),y(2))+2;
    ezplot(M,[xmin,xmax,ymin,ymax])
    hold on
    line([x(1),z(1),y(1)],[x(2),z(2),y(2)],'Color','r')
    plot(x(1),x(2),'Marker','o')
    text(x(1)+.15,x(2),'x');
    plot(y(1),y(2),'Marker','o')
    text(y(1)+.15,y(2),'y');
    plot(z(1),z(2),'Marker','o')
    text(z(1)+.15,z(2),'z');
    legend('Plane of Refraction','Refraction Path x->z->y')
    title('Refraction on Plane Mirror')
end
incidence_angle=(atan(-1/m)-atan((x(2)-z(2))/(x(1)-z(1))))*180/pi;
refraction_angle=(atan(-1/m)-atan((y(2)-z(2))/(y(1)-z(1))))*180/pi;
if incidence_angle<0
    incidence_angle=incidence_angle+180;
end
if refraction_angle<0
    refraction_angle=refraction_angle+180;
end
incidence_angle
refraction_angle
sind(incidence_angle)/sind(refraction_angle)
cx/cy


### `plane-refraction/refraction.m`

In [ ]:
%%writefile {WORK / 'plane-refraction' / 'refraction.m'}
% Reconstructed from the assignment2 source-code PDF.
% Problem-solving approach is retained; see the assignment README.
% Travel-time objective using the speeds on each side of the interface.
function t=refraction(z)
    global x y cx cy
    t=(norm(x-z,2))/cx+(norm(y-z,2))/cy;


## Octave 11.3.0 test results

All six entry-point scripts completed with exit code 0 using Octave `optim` 1.6.3. The captured cell output contains numerical stationary points and solver exit flags. Octave may use different solver behavior and plotting APIs from MATLAB; the suspicious cylindrical maximum-objective formula remains unchanged and is documented in the assignment README.

In [ ]:
import subprocess

def run_case(label, folder, script, needs_optim=True):
    setup = "set(0,'defaultfigurevisible','off');"
    if needs_optim:
        setup += "pkg load optim;"
    setup += "if any(strcmp(available_graphics_toolkits(),'gnuplot')), graphics_toolkit('gnuplot'); end;"
    setup += f"run('{script}');"
    result = subprocess.run(
        [OCTAVE, "--no-gui", "--quiet", "--eval", setup],
        cwd=WORK / folder, text=True, capture_output=True, errors="replace"
    )
    status = "PASS" if result.returncode == 0 else "FAILED"
    print(f"\n=== {label}: {status} (exit code {result.returncode}) ===")
    if result.stdout:
        print(result.stdout, end="" if result.stdout.endswith("\n") else "\n")
    if result.stderr:
        print("--- stderr ---")
        print(result.stderr, end="" if result.stderr.endswith("\n") else "\n")
    return result.returncode

run_case('Circular reflection geometry', 'reflection-geometry', 'geometry1.m', False)
run_case('Plane mirror', 'plane-mirror', 'Reflection_PlaneMirror_2D.m', True)
run_case('Spherical mirror', 'spherical-mirror', 'Reflection_SphericalMirror.m', True)
run_case('Cylindrical mirror', 'cylindrical-mirror', 'Reflection_Cylindrical.m', True)
run_case('External cylindrical reflection', 'cylindrical-mirror', 'Reflection_Cylindrical_Extern.m', True)
run_case('Plane refraction', 'plane-refraction', 'refraction_plane.m', True)



=== Circular reflection geometry: PASS (exit code 0) ===
--- stderr ---

The gnuplot graphics toolkit is not actively maintained and has a number
of limitations that are unlikely to be fixed.  Communication with gnuplot
uses a one-directional pipe and limited information is passed back to the
Octave interpreter so most changes made interactively in the plot window
will not be reflected in the graphics properties managed by Octave.  For
example, if the plot window is closed with a mouse click, Octave will not
be notified and will not update its internal list of open figure windows.
The qt toolkit is recommended instead.

The FLTK graphics toolkit is not actively maintained and has a number
of limitations that are unlikely to be fixed.
The qt toolkit is recommended instead.

=== Plane mirror: PASS (exit code 0) ===
z =

   1.2400
   5.4800

fval = 5.8138
exitflag = 1
incidence_angle = 67.380
reflection_angle = 67.380
--- stderr ---

The gnuplot graphics toolkit is not actively maintaine

In [ ]:
import shutil
shutil.rmtree(WORK)
print("Temporary generated source files removed.")